[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_18/MFM_ch18_quantile_connectedness/MFM_ch18_quantile_connectedness.ipynb)

# MFM_ch18_quantile_connectedness

Case study of Ando, Greenwood-Nimmo and Shin (2022), quantile connectedness of 17 sovereign and 17 financial-sector CDS series, 2006-2017: the published full-sample spillover index of the quantile factor VAR by quantile (Figure 4) against the conditional-mean index, and the published correlations of the rolling tail indices (Table 3) and of the tail indices and relative tail dependence with CATFIN (Figure 13).

*Modelling Financial Markets (MFM), Chapter 18: Systemic Risk and Networks. Author: Daniel Traian Pele.*

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from scipy.stats import spearmanr
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
from sklearn.linear_model import QuantileRegressor, ElasticNetCV
from scipy import optimize
try:
    from arch import arch_model
except ImportError:                      # Colab: the arch package is not preinstalled
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'arch'])
    from arch import arch_model
import warnings
warnings.filterwarnings('ignore')

## Style and helpers

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 11
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['lines.linewidth'] = 1.1
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Course colours
MainBlue = '#1A3A6E'
IDAred   = '#CD0000'
Forest   = '#2E7D32'
Amber    = '#B5853F'
Orange   = '#E67E22'
Purple   = '#8E44AD'
Crimson  = '#DC3545'
Teal     = '#17A2B8'
Navy     = '#1F2A44'   # reference lines (no grey in charts)
BandBlue = '#C5D2E8'   # light MainBlue tint for confidence / reference bands
Gray, LightGray = Navy, BandBlue   # legacy names kept for importing scripts
REGION_COL = {'US': MainBlue, 'EU': IDAred, 'RO': Forest}
REGION_LAB = {'US': 'US banks', 'EU': 'European banks', 'RO': 'Romanian banks (BVB)'}
EVENTS = [('2011-08-08', 'Euro crisis'), ('2016-06-24', 'Brexit vote'), ('2018-12-19', 'RO bank tax'),
          ('2020-03-16', 'COVID-19'), ('2023-03-13', 'SVB'), ('2025-04-07', 'Tariffs')]

HERE = '.'
SEED = 42
B_BOOT = 999
RESULTS = {}


def save_fig(name):
    """Save the figure as transparent PDF and PNG in the current folder."""
    plt.savefig(f'{name}.pdf', bbox_inches='tight', transparent=True)
    plt.savefig(f'{name}.png', bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f"   saved {name}")


def legend_outside_bottom(ax, ncol=2, y=-0.22):
    """Place the legend outside the plot, bottom centre."""
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


def jsonable(x):
    if isinstance(x, dict):
        return {str(k): jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return [jsonable(v) for v in x.tolist()]
    if isinstance(x, (np.floating, np.integer)):
        return x.item()
    if isinstance(x, pd.Timestamp):
        return str(x.date())
    return x


def mark_events(ax, ymax=None):
    """Vertical reference lines for events, with black labels."""
    for d, lab in EVENTS:
        t = pd.Timestamp(d)
        if ax.get_xlim()[0] <= mdates.date2num(t) <= ax.get_xlim()[1]:
            ax.axvline(t, color=Gray, lw=0.5, ls=':')
            ax.text(t, ax.get_ylim()[1] if ymax is None else ymax, lab, rotation=90, fontsize=6.5, color='black',
                    ha='right', va='top')

## Helper functions

In [ ]:
AGS_FIG4 = {0.01: 88.18, 0.05: 77.17, 0.1: 72.2, 0.9: 73.34, 0.95: 79.36, 0.99: 91.77}

AGS_MEAN = 56.57

AGS_CENTRE = (0.3, 0.7, 40.0)

AGS_TAB3 = {'Mean and S(0.95)': 0.774, 'Mean and S(0.05)': 0.411, 'S(0.95) and S(0.05)': 0.367}

AGS_FIG13 = {'S(0.95)': 0.51, 'S(0.05)': 0.15, 'RTD': 0.42}

## Analysis

In [ ]:
def fig_case_quantile_index():
    """Full-sample spillover index by quantile (Figure 4 of the paper)."""
    fig, ax = plt.subplots(figsize=(6.4, 3.0))
    left = [t for t in AGS_FIG4 if t < 0.5]
    right = [t for t in AGS_FIG4 if t > 0.5]
    ax.plot(left, [AGS_FIG4[t] for t in left], color=Forest, marker='o', ms=5, lw=1.3,
            label='Left tail: large beneficial shocks (CDS spreads fall)')
    ax.plot(right, [AGS_FIG4[t] for t in right], color=IDAred, marker='o', ms=5, lw=1.3,
            label='Right tail: large adverse shocks (CDS spreads rise)')
    a, b, c = AGS_CENTRE
    ax.plot([a, b], [c, c], color=Amber, lw=3.0, solid_capstyle='butt',
            label=f'Centre, quantiles {a:.2f} to {b:.2f}: about {c:.0f}%')
    ax.axhline(AGS_MEAN, color=MainBlue, ls='--', lw=1.2, label=f'Conditional mean (VAR): {AGS_MEAN:.2f}%')
    for t, v in AGS_FIG4.items():
        ax.annotate(fr'$\tau$ = {t:.2f}: {v:.2f}%', (t, v), textcoords='offset points',
                    xytext=(8, 0) if t < 0.5 else (-8, 0), ha='left' if t < 0.5 else 'right', va='center',
                    fontsize=7.5, color='black')
    ax.set_xlim(0, 1)
    ax.set_ylim(30, 100)
    ax.set_xticks([0.0, 0.10, 0.30, 0.50, 0.70, 0.90, 1.0])
    ax.set_xlabel(r'Quantile $\tau$ of the QFVAR')
    ax.set_ylabel(r'Spillover index $S_\tau^{(5)}$ (%)')
    ax.set_title('17 sovereigns and 17 financial sectors, daily CDS changes, 2006-2017', fontsize=9.5)
    legend_outside_bottom(ax, ncol=2, y=-0.24)
    save_fig('ch18_case_quantile_index')


def fig_case_tail_corr():
    """Correlations reported in the paper: rolling indices (Table 3) and with CATFIN (Figure 13)."""
    fig, axes = plt.subplots(1, 2, figsize=(7.2, 2.8))
    cols = {'S(0.95)': IDAred, 'S(0.05)': Forest, 'tails': Amber, 'RTD': Purple}
    k3 = list(AGS_TAB3)
    c3 = [cols['S(0.95)'], cols['S(0.05)'], cols['tails']]
    axes[0].bar(range(3), [AGS_TAB3[k] for k in k3], color=c3, width=0.6)
    axes[0].set_xticks(range(3))
    axes[0].set_xticklabels(['Mean,\nS(0.95)', 'Mean,\nS(0.05)', 'S(0.95),\nS(0.05)'])
    axes[0].set_title('Rolling 250-day indices (Table 3)', fontsize=9.5)
    k13 = list(AGS_FIG13)
    axes[1].bar(range(3), [AGS_FIG13[k] for k in k13], color=[cols[k] for k in k13], width=0.6)
    axes[1].set_xticks(range(3))
    axes[1].set_xticklabels(['S(0.95)', 'S(0.05)', 'RTD'])
    axes[1].set_title('Monthly correlation with CATFIN (Figure 13)', fontsize=9.5)
    for ax, vals in [(axes[0], [AGS_TAB3[k] for k in k3]), (axes[1], [AGS_FIG13[k] for k in k13])]:
        for i, v in enumerate(vals):
            ax.text(i, v + 0.02, f'{v:.3f}' if ax is axes[0] else f'{v:.2f}', ha='center', va='bottom',
                    fontsize=8, color='black')
        ax.set_ylim(0, 0.9)
        ax.set_ylabel('Correlation')
    from matplotlib.patches import Patch
    h = [Patch(color=cols['S(0.95)'], label='Involves the right tail S(0.95): large adverse shocks'),
         Patch(color=cols['S(0.05)'], label='Involves the left tail S(0.05): large beneficial shocks'),
         Patch(color=cols['tails'], label='Right tail against left tail'),
         Patch(color=cols['RTD'], label='Relative tail dependence RTD = S(0.95) - S(0.05)')]
    fig.tight_layout()
    fig.legend(handles=h, loc='upper center', bbox_to_anchor=(0.5, 0.0), ncol=2, frameon=False)
    save_fig('ch18_case_tail_corr')

## Run

In [ ]:
fig_case_quantile_index()
fig_case_tail_corr()
print('RTD (full sample) =', round(AGS_FIG4[0.95] - AGS_FIG4[0.05], 2), '%')

![ch18_case_quantile_index](./ch18_case_quantile_index.png)

Output: `ch18_case_quantile_index.pdf`

![ch18_case_tail_corr](./ch18_case_tail_corr.png)

Output: `ch18_case_tail_corr.pdf`